In [8]:
import requests
from bs4 import BeautifulSoup
import json
from urllib.parse import urljoin
import datetime

In [9]:
BASE_URL = "https://sabudh.org/"

# Helper: Get BeautifulSoup object
def get_soup(url):
    try:
        response = requests.get(url, timeout=10)
        return BeautifulSoup(response.text, "html.parser")
    except:
        return None

In [10]:
# Step 1: Get all internal links
def get_all_links():
    soup = get_soup(BASE_URL)
    links = set()

    for a in soup.find_all("a", href=True):
        href = a["href"]
        full_url = urljoin(BASE_URL, href)

        if "sabudh.org" in full_url:
            links.add(full_url)

    return list(links)

In [4]:
# Step 2: Categorize page
def categorize(url):
    url = url.lower()

    if "program" in url:
        return "programs"
    elif "project" in url:
        return "projects"
    elif "about" in url:
        return "about"
    elif "contact" in url:
        return "contact"
    else:
        return "others"

In [11]:
# Cleaning functions
def clean_headings(headings):
    cleaned = []
    seen = set()

    for h in headings:
        h = h.strip()

        if len(h) < 4 or h.isdigit():
            continue

        if h.lower() in ["get in touch", "resources", "organization"]:
            continue

        if h not in seen:
            seen.add(h)
            cleaned.append(h)

    return cleaned


def clean_paragraphs(paragraphs):
    cleaned = []
    seen = set()

    for p in paragraphs:
        p = p.strip()

        if len(p) < 40:
            continue

        # Fix merged text issue
        p = p.replace("EligibilityThis", "Eligibility. This")

        if p not in seen:
            seen.add(p)
            cleaned.append(p)

    return cleaned


In [12]:
# Step 3: Scrape page content
def scrape_page(url):
    soup = get_soup(url)

    if not soup:
        return None

    data = {
        "url": url,
        "headings": [],
        "paragraphs": []
    }

    # Extract headings
    for tag in soup.find_all(["h1", "h2", "h3", "h4"]):
        text = tag.text.strip()
        if text:
            data["headings"].append(text)

    # Extract paragraphs
    for p in soup.find_all("p"):
        text = p.text.strip()
        if text:
            data["paragraphs"].append(text)

    # Apply cleaning
    data["headings"] = clean_headings(data["headings"])
    data["paragraphs"] = clean_paragraphs(data["paragraphs"])

    return data

In [13]:
# Step 4: Main scraper
def scrape_site():
    links = get_all_links()

    final_data = {
        "programs": [],
        "projects": [],
        "about": [],
        "contact": [],
        "others": []
    }

    for link in links:
        print("Scraping:", link)

        page_data = scrape_page(link)
        if not page_data:
            continue

        category = categorize(link)
        final_data[category].append(page_data)

    # Add metadata
    final_data["metadata"] = {
        "source": BASE_URL,
        "total_pages_scraped": len(links),
        "scraped_at": str(datetime.datetime.now())
    }

    return final_data

In [14]:
# Step 5: Run + Save JSON
data = scrape_site()

with open("sabudh_clean_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

print("\nClean data saved as sabudh_clean_data.json")

Scraping: https://sabudh.org/data-science-program/
Scraping: https://sabudh.org/mentor/
Scraping: https://sabudh.org/sun-foundation-ludhiana/
Scraping: https://sabudh.org/#content
Scraping: https://sabudh.org/alumni/
Scraping: https://sabudh.org/contact-us/
Scraping: https://sabudh.org/data-analytics-program/
Scraping: https://sabudh.org/webinars/
Scraping: https://sabudh.org/about-us/
Scraping: mailto:admin.info@sabudh.org
Scraping: https://sabudh.org/partner-on-projects/
Scraping: https://sabudh.org/educational-institute/
Scraping: https://sabudh.org/apply-now/
Scraping: https://sabudh.org/our-projects/
Scraping: https://sabudh.org/gurudwara-shri-guru-singh-sabha-rajouri-garden/
Scraping: https://sabudh.org/privacy/
Scraping: https://sabudh.org/aiot-program/
Scraping: https://sabudh.org/sahibzada-zorawar-singh-charitable-computer-and-educational-centre-saket/
Scraping: https://sabudh.org/
Scraping: https://sabudh.org/our-learning-platform/
Scraping: https://sabudh.org/partner/

Clean